# Setup

In [ ]:
!pip install -qU openai

In [ ]:
import string
import re
from openai import OpenAI
from google.colab import userdata


`string` - importuje moduł `string`, który zawiera zbiór użytecznych stałych i klas do operacji na ciągach znaków. Możemy z niego korzystać, gdy potrzebujemy np. listy wszystkich liter alfabetu, cyfr czy znaków interpunkcyjnych. Jest to szczególnie przydatne podczas przetwarzania tekstu, gdy chcemy wykonać operacje takie jak usunięcie znaków specjalnych.

`re` - importuje moduł wyrażeń regularnych (regular expressions). Moduł ten dostarcza narzędzia do zaawansowanego wyszukiwania i manipulowania wzorcami w tekście. Wyrażenia regularne pozwalają na skomplikowane operacje wyszukiwania, które byłyby trudne do wykonania przy użyciu standardowych metod przetwarzania ciągów znaków.

`OpenAI` - importuje klasę `OpenAI` z biblioteki `openai`. Ta klasa służy jako klient API, który umożliwia nawiązywanie połączenia z usługami OpenAI i korzystanie z ich modeli sztucznej inteligencji, takich jak GPT.

In [ ]:
class CFG:
    model = "gpt-4o-mini"

In [ ]:
import getpass

api_key = userdata.get("openaivision")
if not api_key:
    api_key = getpass.getpass("Enter your OpenAIVision API key: ")

client = OpenAI(api_key=api_key)

# Funkcje

In [ ]:
def preprocess_text(text: str):
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text

Ta funkcja `preprocess_text` służy do przetwarzania wstępnego tekstu – przygotowuje tekst do dalszej analizy lub przetwarzania przez modele języka. Funkcja wykonuje trzy podstawowe operacje czyszczenia tekstu:

1. `text = text.lower()` – zamienia wszystkie litery w tekście na małe litery. Jest to ważny krok standaryzacji, który sprawia, że słowa takie jak "Kot", "KOT" i "kot" będą traktowane jednakowo. Dzięki temu algorytmy przetwarzania języka naturalnego nie będą rozróżniać słów tylko ze względu na wielkość liter.

2. `text = text.translate(str.maketrans("", "", string.punctuation))` – usuwa wszystkie znaki interpunkcyjne z tekstu. Wykorzystuje funkcję `translate` wraz z metodą `str.maketrans`, która tworzy tablicę translacji. Parametr `string.punctuation` zawiera wszystkie standardowe znaki interpunkcyjne (jak kropki, przecinki, wykrzykniki itd.). Po tej operacji tekst "Cześć, jak się masz?" zostanie przekształcony na "Cześć jak się masz".

3. `text = re.sub(r"\s+", " ", text).strip()` – normalizuje białe znaki (spacje, tabulatory, znaki nowego wiersza) w tekście. Wykorzystuje wyrażenie regularne `r"\s+"`, które dopasowuje ciągi składające się z jednego lub więcej białych znaków i zamienia je na pojedynczą spację. Następnie funkcja `strip()` usuwa białe znaki z początku i końca tekstu. Dzięki temu teksty zawierające wiele spacji, tabulatorów czy pustych linii będą ujednolicone, a nadmiarowe białe znaki zostaną usunięte.

In [ ]:
def tokenize(text: str):
    return preprocess_text(text).split()

Ta funkcja `tokenize` wykonuje tokenizację tekstu, czyli proces dzielenia tekstu na mniejsze jednostki zwane tokenami. W tym konkretnym przypadku tokenami będą pojedyncze słowa.

Funkcja działa w dwóch etapach:

1. Najpierw wywołuje wcześniej zdefiniowaną funkcję `preprocess_text(text)`, która:
   - Zamienia wszystkie litery na małe
   - Usuwa znaki interpunkcyjne
   - Normalizuje białe znaki (zamienia ciągi spacji, tabulatorów, znaków nowej linii itp. na pojedyncze spacje)
   - Usuwa białe znaki z początku i końca tekstu

2. Następnie na przetworzonym tekście wywołuje metodę `.split()`, która dzieli tekst na listę słów, używając spacji jako separatora.

Tokenizacja jest **fundamentalną operacją w przetwarzaniu języka naturalnego**, ponieważ wiele algorytmów i modeli językowych pracuje właśnie na poziomie tokenów. Dzięki tokenizacji możemy przekształcić tekst w formę, którą łatwiej analizować algorytmicznie. W bardziej zaawansowanych systemach NLP tokenizacja może uwzględniać również wyrazy złożone, wyrażenia idiomatyczne czy inne jednostki językowe, ale w tym przypadku mamy do czynienia z prostą tokenizacją na poziomie pojedynczych słów.

In [ ]:
def retrieve_relevant_chunks(query: str, corpus: list[str], top_n=2):
    query_tokens = set(tokenize(query))
    similarities = []
    for chunk in corpus:
        chunk_tokens = set(tokenize(chunk))
        similarity = len(query_tokens.intersection(chunk_tokens)) / len(
            query_tokens.union(chunk_tokens)
        )
        similarities.append((chunk, similarity))
    similarities.sort(key=lambda x: x[1], reverse=True)
    return [chunk for chunk, _ in similarities[:top_n]]

Ta funkcja `retrieve_relevant_chunks` służy do wyszukiwania najbardziej relewantnych fragmentów tekstu (chunków) w korpusie na podstawie zapytania użytkownika. Jej działanie opiera się na mierzeniu podobieństwa między zapytaniem a każdym fragmentem korpusu.

Przyjrzyjmy się jej działaniu krok po kroku:

1. Najpierw funkcja tokenizuje zapytanie użytkownika i przekształca listę tokenów w zbiór (set), eliminując powtórzenia:
   ```python
   query_tokens = set(tokenize(query))
   ```

2. Tworzy pustą listę `similarities`, która będzie przechowywać pary (fragment tekstu, wartość podobieństwa):
   ```python
   similarities = []
   ```

3. Dla każdego fragmentu w korpusie wykonuje następujące czynności:
   - Tokenizuje fragment i przekształca go w zbiór:
     ```python
     chunk_tokens = set(tokenize(chunk))
     ```
   
   - Oblicza podobieństwo między zapytaniem a fragmentem używając współczynnika Jaccarda:
     ```python
     similarity = len(query_tokens.intersection(chunk_tokens)) / len(query_tokens.union(chunk_tokens))
     ```
     
     Współczynnik Jaccarda to iloraz liczby elementów wspólnych zbiorów (część wspólna) do liczby wszystkich unikalnych elementów (suma zbiorów). Wartość ta zawsze mieści się w przedziale [0,1], gdzie:
     - 0 oznacza brak wspólnych słów
     - 1 oznacza identyczne zbiory słów

   - Dodaje parę (fragment, podobieństwo) do listy:
     ```python
     similarities.append((chunk, similarity))
     ```

4. Sortuje listę par według wartości podobieństwa, w kolejności malejącej:
   ```python
   similarities.sort(key=lambda x: x[1], reverse=True)
   ```
   
   Użyta tutaj funkcja lambda `lambda x: x[1]` wskazuje, że sortowanie ma się odbywać według drugiego elementu każdej pary (indeks 1), czyli według wartości podobieństwa.

5. Zwraca listę `top_n` fragmentów z najwyższym podobieństwem:
   ```python
   return [chunk for chunk, _ in similarities[:top_n]]
   ```
   
   Wykorzystuje tutaj tzw. list comprehension, aby wyodrębnić same fragmenty (bez wartości podobieństwa) z `top_n` pierwszych elementów posortowanej listy. Symbol `_` oznacza, że nie jesteśmy zainteresowani drugą wartością z pary.

Parametr `top_n=2` jest wartością domyślną, co oznacza, że domyślnie funkcja zwróci dwa najbardziej podobne fragmenty. Można to jednak zmienić, przekazując inną wartość podczas wywoływania funkcji.

Funkcja ta jest przykładem prostego algorytmu wyszukiwania informacji opartego na podobieństwie zbiorów słów. Jest to podstawowa implementacja wektorowego modelu wyszukiwania, który nie uwzględnia jednak takich aspektów jak częstotliwość słów, ich znaczenie czy kolejność. W bardziej zaawansowanych systemach można by użyć technik takich jak TF-IDF, Word Embeddings czy transformatory.

In [ ]:
def answer_question_with_context(query: str, corpus: list[str], top_n=2):
    relevant_chunks = retrieve_relevant_chunks(query, corpus, top_n)
    if not relevant_chunks:
        return "I don't have enough information to answer the question."

    context = "\n".join(relevant_chunks)
    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": f"Based on the provided context, answer the following question: {query}\n\nContext:\n{context}",
            },
            {
                "role": "user",
                "content": query,
            },
        ],
        model=CFG.model,
    )

    return chat_completion.choices[0].message.content.strip()


Funkcja `answer_question_with_context` służy do udzielania odpowiedzi na pytania użytkownika z wykorzystaniem dostępnych fragmentów tekstu jako kontekstu. To klasyczny przykład systemu tzw. Question Answering (odpowiadania na pytania), który jest podstawą wielu współczesnych wyszukiwarek i chatbotów.

Funkcja działa następująco:

1. **Pozyskiwanie relewantnych fragmentów**:
   ```python
   relevant_chunks = retrieve_relevant_chunks(query, corpus, top_n)
   ```
   Funkcja zaczyna od wywołania wcześniej zdefiniowanej funkcji `retrieve_relevant_chunks`, która wyszukuje najbardziej pasujące fragmenty tekstu do zapytania użytkownika. Domyślnie wybierane są dwa najlepiej pasujące fragmenty (`top_n=2`), ale można to zmienić, podając inną wartość jako argument.

2. **Sprawdzenie, czy znaleziono odpowiednie fragmenty**:
   ```python
   if not relevant_chunks:
       return "I don't have enough information to answer the question."
   ```
   Jeśli nie znaleziono żadnych pasujących fragmentów (lista jest pusta), funkcja od razu zwraca komunikat informujący, że nie ma wystarczających informacji do udzielenia odpowiedzi.

3. **Łączenie fragmentów w jeden kontekst**:
   ```python
   context = "\n".join(relevant_chunks)
   ```
   Wszystkie znalezione fragmenty są łączone w jeden tekst, oddzielone znakami nowej linii. Ten zbiorczy tekst będzie stanowił kontekst dla modelu językowego.

4. **Tworzenie zapytania do API OpenAI**:
   ```python
   chat_completion = client.chat.completions.create(
       messages=[
           {
               "role": "system",
               "content": f"Based on the provided context, answer the following question: {query}\n\nContext:\n{context}",
           },
           {
               "role": "user",
               "content": query,
           },
       ],
       model = CFG.model,
   )
   ```
   Tutaj następuje właściwe wywołanie API OpenAI z wykorzystaniem wcześniej utworzonego klienta. Składa się ono z:
   - Listy `messages` zawierającej dwie wiadomości:
     - Wiadomość systemowa (`"role": "system"`), która instruuje model, aby odpowiedział na pytanie na podstawie dostarczonego kontekstu. Zawiera ona zarówno pytanie, jak i zgromadzony wcześniej kontekst.
     - Wiadomość użytkownika (`"role": "user"`), która zawiera samo pytanie.
   - Określenia modelu, który ma być użyty do generowania odpowiedzi (pobierany z konfiguracji `CFG.model`, wcześniej ustawiony jako "gpt-4o-mini").

5. **Wyodrębnienie i zwrócenie odpowiedzi**:
   ```python
   return chat_completion.choices[0].message.content.strip()
   ```
   Z otrzymanej odpowiedzi od API wyodrębniany jest tekst odpowiedzi (pierwsza wygenerowana odpowiedź, jeśli by ich było więcej). Funkcja `strip()` usuwa ewentualne białe znaki z początku i końca odpowiedzi.

Cała ta funkcja implementuje podejście zwane "Retrieval-Augmented Generation" (RAG), które łączy wyszukiwanie relewantnych informacji z generatywnym modelem języka. Dzięki temu model otrzymuje kontekst, który pomaga mu udzielić bardziej precyzyjnej i trafnej odpowiedzi, zamiast polegać wyłącznie na swojej wewnętrznej wiedzy.

Ta metoda ma kilka zalet:
- Pozwala modelowi odpowiadać na podstawie konkretnych, aktualnych informacji
- Umożliwia dostęp do specyficznych danych, których model mógł nie znać
- Ogranicza zjawisko "halucynacji" (wymyślania faktów przez model)
- Zwiększa wiarygodność odpowiedzi, ponieważ są one oparte na dostarczonym materiale źródłowym

In [ ]:
def answer_question(query):
    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": f"Answer the following question: {query}",
            },
            {
                "role": "user",
                "content": query,
            },
        ],
        model=CFG.model,
    )

    return chat_completion.choices[0].message.content.strip()

Ta funkcja `answer_question` to uproszczona wersja poprzedniej funkcji `answer_question_with_context`. W przeciwieństwie do poprzedniej wersji, ta funkcja nie wykorzystuje korpusu tekstowego ani nie wyszukuje relewantnych fragmentów - zamiast tego bezpośrednio przekazuje pytanie do modelu językowego OpenAI.

Rozłóżmy jej działanie na elementy składowe:

1. **Definicja funkcji z parametrami**:
   ```python
   def answer_question(query):
   ```
   Funkcja przyjmuje dwa parametry:
   - `query` - pytanie użytkownika, na które ma zostać udzielona odpowiedź

2. **Wywołanie API OpenAI**:
   ```python
   chat_completion = client.chat.completions.create(
   messages=[
           {
               "role": "system",
               "content": f"Answer the following question: {query}",
           },
           {
               "role": "user",
               "content": query,
           },
       ],
       model = CFG.model,
   )
   ```
   
   W tym kroku funkcja tworzy zapytanie do API OpenAI, które składa się z:
   - Dwóch wiadomości w tablicy `messages`:
     - Wiadomość systemowa (`"role": "system"`), która zawiera instrukcję dla modelu, aby odpowiedział na pytanie
     - Wiadomość użytkownika (`"role": "user"`), która zawiera samo pytanie
   - Określenia modelu, który ma być użyty (pobierany z konfiguracji `CFG.model`, wcześniej ustawiony jako "gpt-4o-mini")

3. **Wyodrębnienie i zwrócenie odpowiedzi**:
   ```python
   return chat_completion.choices[0].message.content.strip()
   ```
   Z otrzymanej odpowiedzi od API wyodrębniany jest tekst odpowiedzi (pierwsza wygenerowana odpowiedź). Funkcja `strip()` usuwa ewentualne białe znaki z początku i końca tekstu.

Warto zwrócić uwagę na kilka istotnych różnic w porównaniu z poprzednią funkcją:

1. **Brak wykorzystania korpusu** - ta funkcja nie korzysta z żadnego zewnętrznego źródła informacji, polega wyłącznie na wiedzy wbudowanej w model językowy.

2. **Uproszczona instrukcja** - instrukcja systemowa jest znacznie prostsza, nie zawiera wzmianki o kontekście, co daje modelowi większą swobodę w formułowaniu odpowiedzi na podstawie własnej wiedzy.

Taka funkcja jest przydatna, gdy:
- Nie mamy specyficznego korpusu tekstowego, na którym chcemy bazować odpowiedzi
- Pytanie dotyczy ogólnej wiedzy, którą model prawdopodobnie posiada
- Zależy nam na szybkości działania (pominięcie etapu wyszukiwania relewantnych fragmentów przyspiesza proces)
- Chcemy uzyskać bardziej ogólne lub kreatywne odpowiedzi, nieograniczone do konkretnego zestawu dokumentów

Jest to podejście typowe dla klasycznych chatbotów opartych na dużych modelach językowych, gdzie odpowiedzi generowane są na podstawie ogólnej wiedzy modelu, a nie z wykorzystaniem specyficznych zewnętrznych źródeł informacji.

# Minimalny RAG

In [ ]:
# minimal corpus
corpus = [
    "The phosphorescent valleys of Lumaria are home to drifting Starwisps that radiate gentle, rhythmic waves of azure light as they float through the mystical terrain.",
    "In the aurora-filled atmosphere of Lumaria, Nebula Moths dance elegantly, their luminous wings scattering cosmic dust across the celestial expanse.",
    "Quantum Elders, sentient crystal formations, share memories through the psionic web that encircles all of Lumaria, preserving ancient secrets and stories.",
    "Within the resonant chambers of Lumaria, Echo Blossoms vibrate with harmonic frequencies that cascade through the mineral halls, orchestrating an alien melody.",
    "Beside the luminous cascades of Lumaria, Prism Dragonflies drift serenely, their diaphanous wings splitting light into spectacular chromatic displays.",
    "Void Serpents, wielders of dimensional energy, glide along the vertical plateaus of Lumaria, bending spatial laws with their mysterious powers.",
    "Stellar Wyrms, magnificent beings of condensed starlight, weave through the iridescent valleys of Lumaria, leaving trails of celestial auroras in their passage.",
    "Through the opalescent beaches of Lumaria, Spectrum Shells crawl and weave, their crystalline bodies transforming sunlight into cascading rainbow patterns.",
]

In [ ]:
corpus = [preprocess_text(sentence) for sentence in corpus]

In [ ]:
question1 = "What are  Spectrum Shells?"
print(f"Question: {question1}")


Question: What are  Spectrum Shells?


In [ ]:
answer1 = answer_question(question1)
print(answer1)


Spectrum shells are a concept in physics and materials science, particularly in the study of atomic and molecular spectra. They refer to layers or "shells" of energy levels that electrons occupy around an atomic nucleus. Each shell corresponds to a specific range of energy levels associated with the quantum states of electrons.

In more detail:

1. **Electron Shells**: In atoms, electrons exist in discrete energy levels, known as shells. These shells are usually designated by principal quantum numbers (n = 1, 2, 3, etc.) and are associated with different energy states.

2. **Spectrum**: The term "spectrum" typically refers to the range of electromagnetic radiation, including visible light, emitted or absorbed by substances. When an electron transitions between energy levels (or shells), it can emit or absorb a photon, resulting in a spectral line.

3. **Role in Spectroscopy**: Spectrum shells are crucial for understanding spectroscopic techniques, which analyze the light emitted or abs

In [ ]:

answer2 = answer_question_with_context(question1, corpus)
print(answer2)

Spectrum Shells are crystalline creatures found on the opalescent beaches of Lumaria. They possess the unique ability to transform sunlight into cascading rainbow patterns, creating a visually captivating display as they crawl and weave along the shoreline. Their appearance and interactions with light contribute to the enchanting atmosphere of Lumaria's vibrant landscape.


In [ ]:
question1 = "Tell me about Lumaria"
print(f"Question: {question1}")


Question: Tell me about Lumaria


In [ ]:
answer1 = answer_question(question1)
print(answer1)

As of my last update in October 2023, there isn't widely recognized information regarding "Lumaria" in terms of a notable concept, entity, or significant cultural reference. However, "Lumaria" could refer to various things, including products, fictional worlds in literature or games, or even names of companies or initiatives. If you provide more context about what specifically you're referring to, I would be glad to help you with more detailed information.


In [ ]:
answer2 = answer_question_with_context(question1, corpus)
print(answer2)

Lumaria is a fantastical and ethereal environment set within the aurora-filled atmosphere of the Lumaria Nebula. This celestial realm is characterized by its vibrant and luminous aesthetics, where moths with glowing wings gracefully flit about, scattering cosmic dust that adds to the beauty of the space. In addition to the moths, Lumaria is home to prism dragonflies, which possess diaphanous wings that can refract light, creating stunning displays of color as they glide through the air. The overall ambiance of Lumaria is one of serenity and wonder, marked by enchanting sights and the interplay of light and cosmic elements. This setting evokes a sense of magical beauty in the vastness of the universe.
